# 11 — Grounded Answer Generation with Citations (Milestone M4 exit)

**DSML stage:** modeling (generation). Closes the PoC loop: question → hybrid retrieval (notebook 10) →
**cited answer** whose every claim traces to an `EvidenceSpan` and its SEC.gov URL.

Grounding rules (per the feasibility studies — provenance failures cause ~80% of perceived quality loss):
- The LLM sees **only** retrieved context; it must cite `[chunk_id]` after every factual sentence
- Claims it cannot support from context must be omitted or flagged as *not in the filings*
- A post-check verifies every cited id exists in the retrieved set (**no citation hallucination**)

**M4 exit criterion:** \"Who are Nvidia's disclosed suppliers and what risks affect them?\" answered with
correct, clickable citations.

In [1]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from litellm import completion

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))
driver.verify_connectivity()

def load_embedder(name: str) -> SentenceTransformer:
    """Load from the local Hugging Face cache (no network); download only if truly absent."""
    try:
        return SentenceTransformer(name, local_files_only=True)
    except OSError:
        print(f"{name} not in the local cache — downloading once...")
        return SentenceTransformer(name)

model = load_embedder(os.getenv("EMBEDDING_MODEL", "Qwen/Qwen3-Embedding-0.6B"))
LLM_MODEL = os.environ["LLM_MODEL"]
CANONICAL = json.loads((PROJECT_ROOT / "artifacts/canonical_entities.json").read_text())

# --- import the hybrid retriever pattern from notebook 10 (duplicated here so each notebook runs standalone;
#     Phase 7 moves this into semigraph.retrieval) ---
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
ALIAS_RES = [
    (re.compile(rf"\b{re.escape(a)}\b", re.I), (name, spec["entity_id"]))
    for name, spec in CANONICAL.items() for a in {name, *spec["aliases"]}
]

def run_cypher(query: str, **params) -> list[dict]:
    with driver.session() as session:
        return [dict(r) for r in session.run(query, **params)]

def embed_query(q: str) -> list[float]:
    # Qwen3-Embedding has a built-in 'query' prompt; bge-style models need the manual prefix.
    if "query" in (model.prompts or {}):
        return model.encode([q], prompt_name="query", normalize_embeddings=True)[0].tolist()
    return model.encode([BGE_QUERY_PREFIX + q], normalize_embeddings=True)[0].tolist()

def hybrid_retrieve(question: str, k_chunks: int = 8, hops: int = 2) -> dict:
    anchors = {name: eid for pat, (name, eid) in ALIAS_RES if pat.search(question)}
    anchor_ids = list(anchors.values()) or [1045810]
    edges = run_cypher(
        f"""MATCH (a:Company) WHERE a.cik IN $ids
        MATCH p = (a)-[r:SUPPLIES_TO|DEPENDS_ON|CUSTOMER_OF|COMPETES_WITH*1..{hops}]-(b:Company)
        UNWIND relationships(p) AS rel
        RETURN DISTINCT startNode(rel).name AS source, type(rel) AS relation, endNode(rel).name AS target,
               rel.evidence_quote AS quote, rel.evidence_chunk_ids AS chunk_ids""",
        ids=anchor_ids)
    risks = run_cypher(
        """MATCH (a:Company)-[:DISCLOSES_RISK {status:'Active'}]->(rf:RiskFactor)-[he:HAS_EVIDENCE]->(e:EvidenceSpan)
        WHERE a.cik IN $ids
        CALL db.index.vector.queryNodes('risk_embedding', 25, $vec) YIELD node, score
        WITH a, rf, he, e, score WHERE node = rf
        RETURN a.name AS company, rf.summary AS summary, rf.category AS category,
               e.chunk_id AS chunk_id, score ORDER BY score DESC LIMIT 6""",
        ids=anchor_ids, vec=embed_query(question))
    chunks = run_cypher(
        """CALL db.index.vector.queryNodes('evidence_embedding', 40, $vec) YIELD node, score
        MATCH (node)-[:MENTIONS]->(c:Company) WHERE c.cik IN $ids
        RETURN DISTINCT node.chunk_id AS chunk_id, score, node.text AS text, node.source_url AS source_url
        ORDER BY score DESC LIMIT $k""",
        ids=anchor_ids, vec=embed_query(question), k=k_chunks)
    return {"anchors": anchors, "edges": edges, "risks": risks, "chunks": chunks}
print("ready")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

ready


## 1. Context assembly + the answering prompt

In [2]:
ANSWER_PROMPT = """You are a semiconductor supply-chain analyst. Answer the question using ONLY the context below,
which was retrieved from SEC filings via a knowledge graph.

Rules:
- Cite evidence after every factual sentence using [chunk_id] (the ids appear in the context).
- Graph relationships (KNOWN RELATIONSHIPS section) may be cited via their chunk ids.
- If the context does not support part of the question, say so plainly — never fill gaps from memory.
- Be concise and structured. Use bullet lists for enumerations.

QUESTION: {question}

=== KNOWN RELATIONSHIPS (from the knowledge graph) ===
{edges_block}

=== DISCLOSED RISKS (semantically ranked) ===
{risks_block}

=== SOURCE EXCERPTS ===
{chunks_block}
"""

def build_context(r: dict) -> tuple[str, str, str, set[str]]:
    valid_ids = set()
    edge_lines = []
    for e in r["edges"]:
        ids = e["chunk_ids"] or []
        valid_ids.update(ids)
        edge_lines.append(f"- {e['source']} {e['relation']} {e['target']} "
                          f"(evidence: \"{e['quote']}\" {' '.join('[' + i + ']' for i in ids)})")
    risk_lines = []
    for k in r["risks"]:
        valid_ids.add(k["chunk_id"])
        risk_lines.append(f"- ({k['category']}) {k['summary']} [{k['chunk_id']}]")
    chunk_lines = []
    for c in r["chunks"]:
        valid_ids.add(c["chunk_id"])
        chunk_lines.append(f"[{c['chunk_id']}]\n{c['text']}\n")
    return ("\n".join(edge_lines) or "(none)", "\n".join(risk_lines) or "(none)",
            "\n".join(chunk_lines) or "(none)", valid_ids)

CITATION_RE = re.compile(r"\[([0-9\-]+:[IVX]+\.[0-9A-Z]+:[0-9]{4})\]")

def answer(question: str) -> dict:
    r = hybrid_retrieve(question)
    edges_block, risks_block, chunks_block, valid_ids = build_context(r)
    prompt = ANSWER_PROMPT.format(question=question, edges_block=edges_block,
                                  risks_block=risks_block, chunks_block=chunks_block)
    # Sonnet 5: no sampling params (400) + thinking disabled (on by default, can return content=None on capped calls)
    resp = completion(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}],
                      max_tokens=1200, thinking={"type": "disabled"})
    text = resp.choices[0].message.content or ""
    assert text, f"empty answer from LLM (finish_reason={resp.choices[0].finish_reason})"
    cited = set(CITATION_RE.findall(text))
    url_by_id = {c["chunk_id"]: c["source_url"] for c in r["chunks"]}
    for cid in cited - set(url_by_id):
        hit = run_cypher("MATCH (e:EvidenceSpan {chunk_id:$i}) RETURN e.source_url AS u", i=cid)
        if hit:
            url_by_id[cid] = hit[0]["u"]
    return {"question": question, "answer": text, "cited": cited, "valid_ids": valid_ids,
            "hallucinated_citations": cited - valid_ids, "citation_urls": url_by_id, "retrieval": r}

## 2. M4 exit question

In [3]:
result = answer("Who are Nvidia's disclosed suppliers and manufacturing partners, and what risks affect them?")
print(result["answer"])
print("\n--- citations ---")
for cid in sorted(result["cited"]):
    print(f"[{cid}] -> {result['citation_urls'].get(cid, '?')}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=3, column=9, offset=143>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 143, 'line': 3, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (a:Company)-[:DISCLOSES_RISK {status:'Active'}]->(rf:RiskFactor)-[he:HAS_EVIDENCE]->(e:EvidenceSpan)\n        WHERE a.cik IN $ids\n        CALL db.index.vector.queryNodes('risk_embedding', 25, $vec) YIELD node, score\n        WITH a, rf, he, e, score WHERE node = rf\n        RETURN a.name AS company, rf.summary AS summary, rf.categ

## Nvidia's Disclosed Suppliers and Manufacturing Partners

**Wafer Foundries:**
- TSMC (Taiwan Semiconductor Manufacturing Company) — used to produce semiconductor wafers [0001045810-26-000021:I.1:0320]
- Samsung Electronics — also used as a wafer foundry [0001045810-26-000021:I.1:0320]

**Memory Suppliers:**
- SK Hynix Inc. [0001045810-26-000021:I.1:0320]
- Micron Technology, Inc. [0001045810-26-000021:I.1:0320]
- Samsung (also a memory supplier, in addition to its foundry role) [0001045810-26-000021:I.1:0320]

**Assembly, Test, and Packaging Subcontractors:**
- Hon Hai Precision Industry Co., Ltd. (Foxconn) [0001045810-26-000021:I.1:0320]
- Wistron Corporation [0001045810-26-000021:I.1:0320]
- Fabrinet [0001045810-26-000021:I.1:0320]

Notably, Samsung occupies a dual role as both a supplier/foundry partner and a disclosed competitor [0001045810-26-000021:I.1:0320][0001045810-26-000021:I.1:0321].

## Risks Affecting These Suppliers and Partners

- **Geographic concentration risk:** N

In [4]:
# Two more probes: an answerable temporal-ish question and an unanswerable one (refusal behavior)
r2 = answer("What does Nvidia say about export controls affecting sales to China?")
print(r2["answer"][:800])
print("\n================\n")
r3 = answer("What is Apple's total headcount?")  # not in the Nvidia PoC corpus — must be declined
print(r3["answer"][:400])

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=3, column=9, offset=143>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 143, 'line': 3, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (a:Company)-[:DISCLOSES_RISK {status:'Active'}]->(rf:RiskFactor)-[he:HAS_EVIDENCE]->(e:EvidenceSpan)\n        WHERE a.cik IN $ids\n        CALL db.index.vector.queryNodes('risk_embedding', 25, $vec) YIELD node, score\n        WITH a, rf, he, e, score WHERE node = rf\n        RETURN a.name AS company, rf.summary AS summary, rf.categ

## Nvidia's Disclosures on Export Controls Affecting China Sales

### Current State: Effective Foreclosure from China
- Nvidia states it is "currently effectively foreclosed from the China market" due to U.S. export controls [0001045810-26-000021:I.1A:0357].
- Under current rules, Nvidia "cannot deliver a competitive product for China's data center market approved by both the US and Chinese governments" [0001045810-26-000021:I.1:0323].

### History of Escalating Restrictions
- **August 2022**: USG announced licensing requirements affecting exports to China (including Hong Kong/Macau) and Russia of A100 and H100 integrated circuits and related systems [0001045810-25-000023:II.7:0272].
- **October 2023**: New licensing requirements took effect covering China and Country Groups D1/D4/D5 (e.g.


No context was provided to answer this question. The knowledge graph returned no relationships, disclosed risks, or source excerpts related to Apple's headcount, so I cannot provide an answer based

In [5]:
# --- M4 EXIT assertion cell ---
assert len(result["cited"]) >= 3, "answer has too few citations"
assert not result["hallucinated_citations"], f"cited ids not in retrieved context: {result['hallucinated_citations']}"
assert "TSMC" in result["answer"], "expected TSMC in Nvidia's supplier answer"
assert all(u and u.startswith("https://www.sec.gov") for u in
           (result["citation_urls"].get(c) for c in result["cited"])), "citations must resolve to SEC URLs"
assert not r2["hallucinated_citations"]
driver.close()
print("M4 COMPLETE — Nvidia PoC answers with verified, SEC-linked citations end-to-end")

M4 COMPLETE — Nvidia PoC answers with verified, SEC-linked citations end-to-end
